# Titanic Survivors — Feature Engineering & Train/Validation Split

This notebook covers:
1. Loading the raw Titanic datasets
2. Exploratory Data Analysis (EDA)
3. Feature extraction and engineering
4. Splitting the training data into an 80% train / 20% validation set

## 1  Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

%matplotlib inline
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 2  Load Data

> Place `train.csv`, `test.csv`, and `gender_submission.csv` inside the `data/` folder.  
> See `data/README.md` for download instructions.

In [ ]:
train_raw = pd.read_csv('data/train.csv')
test_raw  = pd.read_csv('data/test.csv')

print(f'Training set : {train_raw.shape[0]} rows × {train_raw.shape[1]} columns')
print(f'Test set     : {test_raw.shape[0]} rows × {test_raw.shape[1]} columns')
train_raw.head()

## 3  Exploratory Data Analysis

In [ ]:
train_raw.info()

In [ ]:
train_raw.describe(include='all')

In [ ]:
# Missing-value summary
missing = train_raw.isnull().sum().rename('missing').to_frame()
missing['pct'] = (missing['missing'] / len(train_raw) * 100).round(2)
missing[missing['missing'] > 0].sort_values('pct', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Survival rate
train_raw['Survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#d9534f', '#5cb85c'])
axes[0].set_title('Survival Counts')
axes[0].set_xticklabels(['Died (0)', 'Survived (1)'], rotation=0)

# Survival by Sex
train_raw.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[1], color=['#5bc0de', '#f0ad4e'])
axes[1].set_title('Survival Rate by Sex')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

# Survival by Pclass
train_raw.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_title('Survival Rate by Pclass')
axes[2].set_ylabel('Survival Rate')
axes[2].set_xticklabels(['1st', '2nd', '3rd'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Age distribution
train_raw['Age'].plot(kind='hist', bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')

# Fare distribution
train_raw['Fare'].plot(kind='hist', bins=40, ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Fare Distribution')
axes[1].set_xlabel('Fare')

plt.tight_layout()
plt.show()

## 4  Feature Engineering

We apply the same transformations to both the training and test sets to avoid data leakage and ensure consistency.

| Step | Action |
|------|--------|
| `Title` | Extract title from passenger name as a social-status proxy |
| `FamilySize` | `SibSp + Parch + 1` — total people in family unit |
| `IsAlone` | Binary flag: 1 if travelling alone, else 0 |
| `Age` | Impute missing values with the median age per title group |
| `Fare` | Impute one missing test value with the overall median |
| `Embarked` | Impute two missing values with the mode (`S`) |
| Encoding | Label-encode `Sex`; one-hot-encode `Embarked` and `Title` |
| Drop | Remove `PassengerId`, `Name`, `Ticket`, `Cabin` (high cardinality / mostly missing) |

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply feature engineering to a Titanic dataframe."""
    df = df.copy()

    # --- Title -----------------------------------------------------------
    df['Title'] = df['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
    # Consolidate rare titles
    rare_titles = {'Dona', 'Lady', 'Countess', 'the Countess', 'Capt', 'Col', 'Don',
                   'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer'}
    df['Title'] = df['Title'].replace(list(rare_titles), 'Rare')
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

    # --- Family size -----------------------------------------------------
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

    # --- Impute Age (median per Title, with global-median fallback) ------
    title_age_medians = df.groupby('Title')['Age'].median()
    df['Age'] = df['Age'].fillna(df['Title'].map(title_age_medians))
    df['Age'] = df['Age'].fillna(df['Age'].median())

    # --- Impute Fare & Embarked ------------------------------------------
    df['Fare']     = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    # --- Encode Sex ------------------------------------------------------
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

    # --- One-hot encode Embarked & Title ---------------------------------
    # drop_first=False keeps all categories; tree-based models are unaffected
    # by multicollinearity. Use drop_first=True for linear models.
    df = pd.get_dummies(df, columns=['Embarked', 'Title'], drop_first=False)

    # --- Drop columns not used as features ------------------------------
    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    return df


train_eng = engineer_features(train_raw)
test_eng  = engineer_features(test_raw)

# Align test columns to training columns (excluding target)
feature_cols = [c for c in train_eng.columns if c != 'Survived']
# Warn if test set has categories not seen during training
unseen = [c for c in test_eng.columns if c not in feature_cols]
if unseen:
    print(f'Warning: test set has unseen columns that will be dropped: {unseen}')
test_eng = test_eng.reindex(columns=feature_cols, fill_value=0)

print('Engineered training set shape :', train_eng.shape)
print('Engineered test set shape     :', test_eng.shape)
train_eng.head()

In [ ]:
# Confirm no missing values remain in training set
missing_after = train_eng.isnull().sum()
print('Columns still containing NaN:', missing_after[missing_after > 0].to_dict() or 'None ✓')

## 5  Train / Validation Split (80 / 20)

We use a stratified split to preserve the class distribution.

In [ ]:
X = train_eng.drop(columns=['Survived'])
y = train_eng['Survived']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'X_val   : {X_val.shape}   |  y_val   : {y_val.shape}')
print()
print('Survival rate — train :', round(y_train.mean(), 4))
print('Survival rate — val   :', round(y_val.mean(), 4))
print()
print('Features used:')
print(list(X.columns))

## 6  Feature Correlation Heatmap

In [ ]:
corr_df = pd.concat([X_train, y_train], axis=1)
plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_df.corr(),
    annot=True, fmt='.2f', linewidths=0.5,
    cmap='coolwarm', center=0
)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## Summary

| Dataset | Rows | Features |
|---------|------|----------|
| Full training (raw) | 891 | 12 |
| Engineered training | 891 | varies |
| **X_train** | ~713 | — |
| **X_val** | ~178 | — |
| Test (Kaggle) | 418 | — |

The splits `X_train`, `X_val`, `y_train`, `y_val` and the Kaggle test features `test_eng` are ready for model training in a follow-up notebook.